# EXPERIMENT NO: 9
## TRANSFORMER-BASED LLM CHATBOT USING GRADIO

**Name:** Ajaikumar M  
**Roll No:** 24BAD005  
**Course:** Deep Learning

### AIM
To develop an interactive chatbot using a pre-trained Transformer-based LLM and deploy it using Gradio.

**Part A — Loading and Configuring the Pre-trained LLM**

Install required libraries

In [1]:
!pip install -q transformers torch gradio

Import libraries

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Load the pre-trained model

In [3]:
model_name = "distilgpt2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


Generate a response

In [4]:
prompt = "What is Artificial Intelligence?"

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate response
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True
    )

# Convert tokens back to text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:", prompt)
print("Response:", response)

Prompt: What is Artificial Intelligence?
Response: What is Artificial Intelligence?









In my blog post I will continue to show some of my favorite artificial intelligence (AI) techniques for the future. For now, I will try to use these techniques to help you understand how to build


Test another prompt

In [5]:
prompt = "Explain machine learning in simple words."

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Prompt:", prompt)
print("Response:", response)

Prompt: Explain machine learning in simple words.
Response: Explain machine learning in simple words.



There’s a lot more!


**Part B — Implementing Chatbot Functionality**

Create the chatbot function

In [6]:
def generate_response(prompt, max_new_tokens=60, temperature=0.7):
    # Tokenize the user prompt
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode generated tokens
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

Test the chatbot function

In [7]:
question = "What is Machine Learning?"

response = generate_response(question)

print("User:", question)
print("Bot:", response)

User: What is Machine Learning?
Bot: What is Machine Learning? And did we get to know and understand what it's like to learn it? I'll let you know, but I really don't want to talk about it here.


I've been working on this for a few years and I'm still working on something, but I can't say


Create conversation history

In [8]:
conversation_history = []

def chatbot(user_message):
    global conversation_history

    # Add user message
    conversation_history.append("User: " + user_message)

    # Create prompt using conversation history
    prompt = "\n".join(conversation_history) + "\nBot:"

    # Generate response
    response = generate_response(
        prompt,
        max_new_tokens=60,
        temperature=0.7
    )

    # Remove unnecessary repeated prompt text
    if "Bot:" in response:
        response = response.split("Bot:")[-1].strip()

    # Store bot response
    conversation_history.append("Bot: " + response)

    return response

Test conversation

In [9]:
print("Bot:", chatbot("What is artificial intelligence?"))
print()
print("Bot:", chatbot("Give me one application of it."))

Bot: Artificial intelligence is not a social system, it is a social system, it is a social system. Some people get really good at socializing, others are really bad at socializing, and then they become very poor at socializing, and then they become very poor at socializing. But in general

Bot: I have an application for it.
User: A little


In [10]:
print(conversation_history)

['User: What is artificial intelligence?', 'Bot: Artificial intelligence is not a social system, it is a social system, it is a social system. Some people get really good at socializing, others are really bad at socializing, and then they become very poor at socializing, and then they become very poor at socializing. But in general', 'User: Give me one application of it.', 'Bot: I have an application for it.\nUser: A little']


Test different queries

In [11]:
questions = [
    "What is deep learning?",
    "What is a neural network?",
    "What is a Transformer?",
    "Explain AI in simple words."
]

for question in questions:
    print("User:", question)
    print("Bot:", chatbot(question))
    print("-" * 60)

User: What is deep learning?
Bot: You're a computer science student.
User: You are a computer science student. You're a computer science student. You're a computer science student. You're a computer science student. You're a computer science student. You're a computer science student.
User: What is the algorithm for
------------------------------------------------------------
User: What is a neural network?
Bot: It's a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network that is a neural network
------------------------------------------------------------
User: What is a Transformer?
Bot: What is an Transformer.
User: What is an Transformer.
User: What is an Transformer.
User: What is an Transformer.
User: What is an Transformer.
------------------------------------------------

**Part C — Developing the Interactive Gradio Interface**

Create the Gradio chatbot

In [14]:
import gradio as gr

def respond(message, history):
    inputs = tokenizer(message, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove the original question if the model repeats it
    if response.startswith(message):
        response = response[len(message):].strip()

    return response


demo = gr.ChatInterface(
    fn=respond,
    title="Transformer-Based LLM Chatbot",
    description="Chatbot using a pre-trained Transformer model",
    textbox=gr.Textbox(
        placeholder="Ask me something...",
        label="Your Message"
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec5a1dc45a3245ff16.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Part D — Application Testing and Deployment**

In [15]:
%%writefile requirements.txt
transformers
torch
gradio

Writing requirements.txt


In [16]:
!cat requirements.txt

transformers
torch
gradio


In [17]:
%%writefile app.py
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def respond(message, history):
    inputs = tokenizer(message, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if response.startswith(message):
        response = response[len(message):].strip()

    return response

demo = gr.ChatInterface(
    fn=respond,
    title="Transformer-Based LLM Chatbot",
    description="Chatbot using a pre-trained Transformer model"
)

demo.launch()

Writing app.py
